# EXP_060A — Greedy Sequential Best: Full System
**Phase 6 | Promising Combination Validation**
Research question: Does the greedy sequential winner form the best full system?
- All best components from Phases 2-5 combined | Seed: 42 | AMP: enabled
> ⚠️ Fill ALL variables in STEP 4 based on your Phase 2-5 results.

### STEP 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### STEP 2: Clone source code and install dependencies

In [ ]:
!git clone https://github.com/lechihoang/SE365.git
%cd SE365
!pip install -r requirements.txt -q

### STEP 3: Download and extract data

In [ ]:
!rm -rf ./data
!cp /content/drive/MyDrive/SE365/data.zip ./data.zip
!unzip -q data.zip
!rm data.zip
!ls -la ./data

### STEP 4: Configure paths — ✏️ Set ALL Phase 2-5 winners here

In [ ]:
import os
DRIVE_ROOT = '/content/drive/MyDrive/SE365'  # ✏️ Change if needed
EXP_ID = 'EXP_060A_bestsequential'

# ✏️ Fill all from Phase 2-5 winners
BEST_IMAGE_MODEL  = 'swin_base_patch4_window7_224'
BEST_TEXT_MODEL   = 'vinai/phobert-base-v2'
BEST_FUSION_TYPE  = 'gmu'
BEST_LOSS         = 'huber'
BEST_IMAGE_EXP_ID = 'EXP_020B_swinb_xlmr_concat_mse'
BEST_TEXT_EXP_ID  = 'EXP_030B_bestimage_phobert_concat_mse'

DRIVE_EXP_PATH = f'{DRIVE_ROOT}/experiments/{EXP_ID}'
os.makedirs(DRIVE_EXP_PATH, exist_ok=True)
print(f'Artifacts: {DRIVE_EXP_PATH}')
print(f'Config: Image={BEST_IMAGE_MODEL} | Text={BEST_TEXT_MODEL} | Fusion={BEST_FUSION_TYPE} | Loss={BEST_LOSS}')

### STEP 5: Load pretrained weights

In [ ]:
import os, shutil
os.makedirs('./checkpoints', exist_ok=True)
shutil.copy(f'{DRIVE_ROOT}/experiments/{BEST_TEXT_EXP_ID}/best_model_train_text.pth', './checkpoints/best_model_train_text.pth')
print(f'Loaded text from {BEST_TEXT_EXP_ID}')
shutil.copy(f'{DRIVE_ROOT}/experiments/{BEST_IMAGE_EXP_ID}/best_model_train_image.pth', './checkpoints/best_model_train_image.pth')
print(f'Loaded image from {BEST_IMAGE_EXP_ID}')

### STEP 6: Train

In [ ]:
!python main.py \
  --mode train_fusion \
  --fusion_type {BEST_FUSION_TYPE} \
  --text_model_name {BEST_TEXT_MODEL} \
  --image_model_name {BEST_IMAGE_MODEL} \
  --epochs 20 \
  --batch_size 16 \
  --lr 1e-5 \
  --grad_accum_steps 2 \
  --patience 5 \
  --loss_fn {BEST_LOSS} \
  --unfreeze_text_layers 1 \
  --unfreeze_image_layers 1 \
  --seed 42 \
  --use_amp \
  --exp_id EXP_060A_bestsequential \
  --exp_dir ./experiments

### STEP 7: Save to Drive + print metrics

In [ ]:
import json
!cp -r ./experiments/$EXP_ID/* $DRIVE_EXP_PATH/

with open(f'./experiments/{EXP_ID}/metrics.json') as f:
    m = json.load(f)

print(f'\n=== {EXP_ID} Results ===')
print(f"Loss (val)   : {m['loss']:.4f}")
print()
print("             MAE      RMSE      R2")
print(f"  food     : {m['mae_food']:.4f}   {m['rmse_food']:.4f}   {m['r2_food']:.4f}")
print(f"  price    : {m['mae_price']:.4f}   {m['rmse_price']:.4f}   {m['r2_price']:.4f}")
print(f"  atmos    : {m['mae_atmos']:.4f}   {m['rmse_atmos']:.4f}   {m['r2_atmos']:.4f}")
print(f"  service  : {m['mae_service']:.4f}   {m['rmse_service']:.4f}   {m['r2_service']:.4f}")
print(f"  overall  : {m['mae_overall']:.4f}   {m['rmse_overall']:.4f}   {m['r2_overall']:.4f}")
print()
print(f"  mean_mae   : {m['mean_mae']:.4f}")
print(f"  aspect_mae : {m['aspect_mae']:.4f}")
print(f"  overall_mae: {m['overall_mae']:.4f}")